# Arquitetura em Camadas (Layers) — Exemplo Executável

**Guia Aberto de Arquitetura de Software** · [Página do tópico](https://ifpe-belojardim-es.github.io/guia-arquitetura-de-software/content/02-arquitetura-em-camadas/)

**Autores:**
- Maria Letícia de Sousa Barboza
- João Henrique Lopes de Araújo Freire
- Pedro Henrique Reis Xavier
- Wellison Danniel Soares Ponciano
- Caio Vinícius Santana Gomes
- Ruan Henrique Pereira dos Santos

**Licença:** CC BY-NC 4.0

---

Este notebook é o exemplo executável do tópico. Implementa um sistema de
**gerenciamento de tarefas** em três camadas:

- **Persistência** (`TarefaRepository`) — acesso aos dados, simulado em memória. Não valida nada.
- **Domínio** (`TarefaService`) — regras e validações de negócio.
- **Apresentação** (`PresentationCLI`) — recebe a ação do usuário e exibe resultados.

Regra fundamental: **as dependências só apontam para baixo** (Apresentação → Domínio → Persistência). Uma camada **nunca** pode depender da camada acima.

Estrutura:
1. Roda de ponta a ponta em ambiente limpo (sem dependências externas)
2. Demonstra o conceito, não apenas ilustra
3. **Mostra a violação** — uma camada inferior que chama uma camada superior
4. Explica em células de texto, não só em comentários de código

## 0. Dependências

Sem bibliotecas externas — só a biblioteca padrão do Python.

In [ ]:
import sys
import inspect

print(f"Python {sys.version.split()[0]}")

## 1. O problema

Um sistema de gerenciamento de tarefas precisa:

1. **Validar** dados de entrada (título não pode ser vazio, prioridade deve ser `baixa`, `media` ou `alta`).
2. **Persistir** os dados em algum lugar.
3. **Apresentar** o resultado para quem usa o sistema — seja via terminal, seja via API.

Se essas três responsabilidades vivem na mesma classe, qualquer mudança em uma — trocar o banco de dados, adicionar uma nova regra de validação, ou substituir o terminal por uma interface web — arrisca quebrar as outras duas.

O estilo de Arquitetura em Camadas resolve isso com uma regra simples: **as dependências sempre apontam para baixo**. A camada de Apresentação conhece o Domínio; o Domínio conhece a Persistência. O inverso é estritamente proibido. A violação que demonstramos neste notebook é exatamente o inverso: uma camada inferior que tenta chamar uma camada superior.

## 2. Implementação seguindo o estilo

### 2.1 Camada de Persistência (`TarefaRepository`)

Responsável apenas por armazenar e recuperar dados. **Não valida nada, não conhece regras de negócio, não conhece a Apresentação.** É a camada mais baixa da hierarquia.

In [ ]:
class TarefaRepository:
    """Simula um banco de dados em memória.
    Responsabilidade única: salvar, ler, atualizar e remover registros.
    Não decide se um dado é válido — isso é papel do Domínio."""

    def __init__(self):
        self._tarefas = {}
        self._proximo_id = 1

    def salvar(self, dados: dict) -> int:
        tarefa_id = self._proximo_id
        self._tarefas[tarefa_id] = dados
        self._proximo_id += 1
        return tarefa_id

    def buscar_todas(self) -> list:
        return [dict(t, id=tid) for tid, t in self._tarefas.items()]

    def buscar_por_id(self, tarefa_id: int) -> dict | None:
        tarefa = self._tarefas.get(tarefa_id)
        return dict(tarefa, id=tarefa_id) if tarefa else None

    def atualizar(self, tarefa_id: int, dados: dict) -> bool:
        if tarefa_id not in self._tarefas:
            return False
        self._tarefas[tarefa_id] = dados
        return True

    def remover(self, tarefa_id: int) -> bool:
        return self._tarefas.pop(tarefa_id, None) is not None

### 2.2 Camada de Domínio (`TarefaService`)

O "coração" do sistema. Aplica validações — título obrigatório e prioridade dentro de um conjunto válido. **Depende da Persistência (camada abaixo), mas a Persistência não conhece o Domínio.**

In [ ]:
class TarefaInvalidaError(Exception):
    pass


class TarefaService:
    """Contém as regras de negócio. É a única camada que decide o que é uma
    tarefa válida e o que acontece quando ela é criada, concluída, etc."""

    PRIORIDADES_VALIDAS = {"baixa", "media", "alta"}

    def __init__(self, repositorio: TarefaRepository):
        self._repo = repositorio  # depende da camada abaixo — correto

    def criar_tarefa(self, titulo: str, prioridade: str = "media") -> dict:
        if not titulo or not titulo.strip():
            raise TarefaInvalidaError("O título da tarefa não pode ser vazio.")
        if prioridade not in self.PRIORIDADES_VALIDAS:
            raise TarefaInvalidaError(
                f"Prioridade inválida: {prioridade!r}. Use uma de {self.PRIORIDADES_VALIDAS}."
            )
        dados = {"titulo": titulo.strip(), "prioridade": prioridade, "concluida": False}
        tarefa_id = self._repo.salvar(dados)
        return self._repo.buscar_por_id(tarefa_id)

    def concluir_tarefa(self, tarefa_id: int) -> dict:
        tarefa = self._repo.buscar_por_id(tarefa_id)
        if tarefa is None:
            raise TarefaInvalidaError(f"Tarefa {tarefa_id} não existe.")
        tarefa["concluida"] = True
        self._repo.atualizar(tarefa_id, {k: v for k, v in tarefa.items() if k != "id"})
        return tarefa

    def listar_pendentes(self) -> list:
        return [t for t in self._repo.buscar_todas() if not t["concluida"]]

    def listar_todas_ordenadas(self) -> list:
        ordem = {"alta": 0, "media": 1, "baixa": 2}
        return sorted(self._repo.buscar_todas(), key=lambda t: ordem[t["prioridade"]])

### 2.3 Camada de Apresentação (`PresentationCLI`)

Só traduz a interação do usuário em chamadas ao Domínio e formata a saída. **Não contém nenhuma regra de negócio.** Repare que ela nunca referencia `TarefaRepository` — só fala com `TarefaService`.

In [ ]:
class PresentationCLI:
    """Interface baseada em texto, simulando um terminal.
    Poderia ser trocada por uma interface web sem tocar no Domínio."""

    def __init__(self, service: TarefaService):
        self._service = service  # depende do Domínio (camada abaixo) — correto

    def adicionar(self, titulo: str, prioridade: str = "media"):
        try:
            tarefa = self._service.criar_tarefa(titulo, prioridade)
            print(f"Tarefa criada: #{tarefa['id']} — {tarefa['titulo']} [{tarefa['prioridade']}]")
        except TarefaInvalidaError as erro:
            print(f"Erro: {erro}")

    def concluir(self, tarefa_id: int):
        try:
            self._service.concluir_tarefa(tarefa_id)
            print(f"Tarefa #{tarefa_id} marcada como concluída.")
        except TarefaInvalidaError as erro:
            print(f"Erro: {erro}")

    def exibir_pendentes(self):
        pendentes = self._service.listar_pendentes()
        print("--- Tarefas pendentes ---")
        if not pendentes:
            print("(nenhuma)")
        for t in pendentes:
            print(f"#{t['id']} [{t['prioridade']}] {t['titulo']}")


# Montagem da aplicação respeitando a hierarquia:
# Persistência <- Domínio <- Apresentação
repositorio = TarefaRepository()
servico = TarefaService(repositorio)
tela = PresentationCLI(servico)

tela.adicionar("Estudar arquitetura em camadas", "alta")
tela.adicionar("Revisar slides de HCI", "media")
tela.adicionar("", "alta")            # dispara TarefaInvalidaError (título vazio)
tela.adicionar("Tarefa X", "urgente") # dispara TarefaInvalidaError (prioridade inválida)
print()
tela.exibir_pendentes()

### Por que isso é testável

A consequência prática de respeitar a hierarquia: o Domínio pode ser exercitado sem persistência real, pois `TarefaService` depende apenas da interface pública de `TarefaRepository`, não de um banco de dados de verdade.

In [ ]:
class RepositorioFalso(TarefaRepository):
    """Dublê de teste — mesma interface pública, sem nenhuma infraestrutura
    real por trás. Possível justamente porque o Domínio depende da interface,
    não de uma implementação específica."""
    pass


falso = RepositorioFalso()
TarefaService(falso).criar_tarefa("Testar sem banco de dados", "alta")

assert len(falso.buscar_todas()) == 1
print("Teste passou — Domínio exercitado sem banco de dados real.")

## 3. A violação

O princípio que a Arquitetura em Camadas mais protege é a **direção das dependências**: elas só devem apontar para baixo. A violação que demonstramos aqui é exatamente o inverso: a **Persistência (`TarefaRepository`) tentando chamar o Domínio (`TarefaService`)** — uma dependência ascendente.

Imagine um requisito novo: "ao salvar uma tarefa, o repositório deve notificar o domínio para disparar alguma ação". A solução ingênua é injetar o `TarefaService` dentro do `TarefaRepository`:

```
Persistência → Domínio  ← dependência proibida (aponta para cima)
```

O problema imediato é o **ciclo de dependências**: `TarefaService` já depende de `TarefaRepository`. Se `TarefaRepository` passar a depender de `TarefaService`, os dois passam a depender um do outro — e nenhum pode ser instanciado nem testado sem o outro.

In [ ]:
# ANTIPADRÃO: Persistência → Domínio (dependência ascendente)
#
# TarefaRepositoryViolador injeta TarefaService para poder "notificar"
# o domínio ao salvar. Isso cria um ciclo:
#   TarefaService depende de TarefaRepository
#   TarefaRepositoryViolador passa a depender de TarefaService
#   → ciclo: nenhum dos dois pode ser instanciado sem o outro.

class TarefaRepositoryViolador(TarefaRepository):
    """ANTIPADRÃO: repositório que chama o Domínio — dependência ascendente."""

    def __init__(self, servico_notificacao=None):
        super().__init__()
        # Persistência agora conhece o Domínio — isso é proibido pelo estilo.
        self._servico = servico_notificacao

    def salvar(self, dados: dict) -> int:
        tarefa_id = super().salvar(dados)
        # Chama o Domínio a partir da Persistência — dependência ascendente.
        if self._servico is not None:
            self._servico.listar_pendentes()  # camada inferior invocando camada superior
        return tarefa_id


# Demonstração do ciclo de instanciação:
# Para criar o repositório, precisamos do serviço.
# Para criar o serviço, precisamos do repositório.
# Resolvemos com None + atribuição posterior — uma gambiarra que o estilo evita.
repo_violador = TarefaRepositoryViolador(servico_notificacao=None)
servico_violador = TarefaService(repo_violador)
repo_violador._servico = servico_violador  # atribuição forçada após criação

print("Repositório violador instanciado — mas apenas com gambiarra (None + atribuição posterior).")
print("Consequência: Persistência e Domínio tornaram-se inseparáveis.")
print("Não é possível testar TarefaRepositoryViolador sem carregar TarefaService junto.")

## 4. Consequência mensurável

Em vez de apenas afirmar que `TarefaRepositoryViolador` viola o estilo, medimos isso de forma verificável: inspecionamos o código-fonte de cada método das classes de **Persistência** e contamos quantas vezes citam, por nome, uma classe de **Domínio** (`TarefaService` ou `TarefaInvalidaError`).

Uma implementação de Persistência que respeita o estilo deve ter **zero** referências ao Domínio.

In [ ]:
DOMINIO = {"TarefaService", "TarefaInvalidaError"}

def referencias_dominio(cls) -> dict:
    """Conta, no código-fonte de cada método da classe, quantas vezes
    aparecem nomes de classes de Domínio — que a Persistência não deveria conhecer."""
    encontradas = {}
    for nome, membro in vars(cls).items():
        try:
            codigo = inspect.getsource(membro)
        except (TypeError, OSError):
            continue
        for simbolo in DOMINIO:
            if simbolo in codigo:
                encontradas.setdefault(nome, []).append(simbolo)
    return encontradas


print(f"{'Classe de Persistência':35} | {'Refs ao Domínio':30} | Situação")
print("-" * 80)
for cls in (TarefaRepository, TarefaRepositoryViolador):
    refs = referencias_dominio(cls)
    if refs:
        situacao = "VIOLA o estilo"
        detalhes = ", ".join(f"{m}:{s}" for m, ss in refs.items() for s in ss)
    else:
        situacao = "respeita o estilo"
        detalhes = "nenhuma"
    print(f"{cls.__name__:35} | {detalhes:30} | {situacao}")

## 5. Conclusão

`TarefaRepository` não conhece nada acima de si — pode ser testado, trocado ou reimplementado sem tocar no Domínio. `TarefaRepositoryViolador` demonstra o custo real de uma dependência ascendente: o ciclo de instanciação força gambiarras, Persistência e Domínio tornam-se inseparáveis nos testes, e qualquer mudança em uma classe propaga-se obrigatoriamente para a outra.

A regra "dependências só apontam para baixo" não é burocracia: é a única garantia de que cada camada pode evoluir, ser testada e ser substituída de forma independente.

---

### Referências

[1] F. Buschmann, R. Meunier, H. Rohnert, P. Sommerlad e M. Stal, *Pattern-Oriented Software Architecture: A System of Patterns*. John Wiley & Sons, 1996.

[2] L. Bass, P. Clements e R. Kazman, *Software Architecture in Practice*, 4ª ed. Addison-Wesley, 2021.

[3] I. Sommerville, *Engenharia de Software*, 9ª ed. Pearson, 2011.

[4] M. Fowler, "PresentationDomainDataLayering", *martinfowler.com*, ago. 2015. Disponível em: https://martinfowler.com/bliki/PresentationDomainDataLayering.html

---

Conteúdo sob CC BY-NC 4.0 — uso livre com crédito. Código sob MIT.